**打个比方**：
-   **第一阶段**：学习新知识前，大量阅读各种书籍（未标注的EEG数据），但不要求写读后感。只是通过阅读，学会了语言的基本规律和词汇。
-   **第二阶段**：针对几篇特定的文章（标注的EEG-文本对），练习写读后感（生成文本）。老师（损失函数）会根据读后感与标准答案的差距来指导他改进。

这种“先广泛阅读，再精读写作”的混合范式，是目前让EEG到文本模型既强大又实用的关键。

![alt text](eeg2text.png)

#### EEG2Text
* 第一阶段：自监督预训练，得到通用的EEG特征提取器。将EEG信号转换为一个特征向量,这个向量还在EEG自己的空间里。
* 第二阶段：加入投影器，将EEG特征映射到文本词嵌入空间，翻译成LLM能理解的文本嵌入空间中的向量。
* 第三阶段：有监督微调，将EEG特征映射到文本空间，实现EEG-文本的对齐。

完整训练策略  
```text
Stage 1:
预训练 EEG Encoder

Stage 2:
冻结 EEG Encoder 和 LLM
训练 Projector

Stage 3:
冻结 LLM backbone
训练 Projector + LoRA

Stage 4:
可选：小学习率微调 EEG Encoder 后几层
```

projector 会把 EEG 特征变成 LLM 能接收的 embedding。  
但是 LLM 原本是为文本 token 训练的，不是为 EEG embedding 训练的。即使 projector 把维度对齐了，LLM 也不一定知道怎么利用这些 EEG prefix。  
所以 LoRA 的作用是让 LLM 轻微适配这种新输入


```text
                 ┌────────────────────┐
EEG signal ─────▶│    EEG Encoder      │
                 └─────────┬──────────┘
                           │
                    EEG latent states
                           │
                 ┌─────────▼──────────┐
                 │ Projector / Adapter│
                 └─────────┬──────────┘
                           │
                   EEG prefix embeddings
                           │
Text input ────────────────┼──────────────▶ LLM Decoder ───▶ Text output
                           │
                    Cross-modal alignment
```




### 1. 自监督预训练 —— “打基础”

这个阶段的目标是**让模型学会理解EEG信号本身的语言**，而不需要知道它对应什么文本。得到通用的EEG特征提取器，可以将任意一段EEG信号映射到一个有意义的向量空间中。


##### step1：对比学习（Contrastive Learning）

-   **核心思想**：让模型学会区分“相似”和“不相似”的EEG信号片段。
-   **具体操作**：
    1.  **数据增强**：对同一段EEG信号进行不同的增强处理（如添加少量噪声、时间偏移、通道掩码等），得到两个“正样本对”。
    2.  **构造负样本**：从其他时间点或其他受试者的EEG信号中随机选取片段，作为“负样本”。
    3.  **训练目标**：让模型将正样本对在向量空间中的距离拉近，将负样本对的距离推远。
    4.  **只需要EEG信号本身**，不需要任何文本标注。


##### step2：掩码重建（Masked Reconstruction）

-   **核心思想**：让模型像“完形填空”一样，根据上下文预测被遮盖的EEG片段。
-   **具体操作**：
    1.  **随机掩码**：随机遮盖EEG信号中的一部分时间点或通道。
    2.  **训练目标**：让模型根据未被遮盖的部分，预测被遮盖部分的信号值。
    3.  **模型架构**：通常使用Transformer或CNN+RNN的编码器-解码器结构。


##### 此时的模型具备以下能力：

-   EEG信号的**基本统计规律**（如功率谱密度、时频分布）。
-   EEG信号的**时空结构**（如不同脑区之间的协同活动模式）。
-   EEG信号的**个体差异**（如不同受试者的α波频率可能不同）。
-   一个**通用的EEG特征提取器**，可以将任意一段EEG信号映射到一个有意义的向量空间中。

自监督训练可以先让 encoder 学会 EEG 的基本结构，比如：  
-   信号的时序变化
-   频段特征
-   通道之间的空间关系
-   事件相关电位 ERP
-   不同 trial 的稳定模式
-   噪声和伪迹特征  
这一步不需要文本标签，只需要 EEG 本身。


### 2.共享词嵌入空间

核心原因：让模型学会“猫”和“看到猫的脑电波”是同一回事

想象一下，如果你有两个独立的向量空间：
-   **EEG空间**：所有EEG信号都在这里，向量之间的距离表示EEG波形的相似度。
-   **文本空间**：所有文本都在这里，向量之间的距离表示语义的相似度。

那么，一段“看到猫”的EEG信号和一个“猫”字，它们**永远无法直接比较**。模型不知道EEG空间里的某个点，对应文本空间里的哪个点。这就是所谓的**模态鸿沟（Modality Gap）**。

**将它们映射到同一个共享空间，就是为了填平这个鸿沟。** 在这个共享空间里：
-   “看到猫”的EEG向量
-   “猫”这个字的文本向量
-   “一张猫的图片”的图像向量

它们三者之间的距离，直接反映了**语义上的相似度**。模型可以直观地知道：“哦，这段EEG信号和‘猫’这个字说的是同一件事。”

---

### 原理

#### 1. 实现跨模态检索（Cross-Modal Retrieval）

这是最直接的应用。在共享空间里，你可以：
-   **用EEG搜文本**：输入一段EEG信号，找到最匹配的文本描述。
-   **用文本搜EEG**：输入一句文本，找到历史上对应的EEG信号片段。
-   **用EEG搜图片**：输入一段EEG信号，找到受试者当时正在看的图片。

**如果不共享空间，这些操作根本无法实现。**

#### 2. 实现零样本学习（Zero-Shot Learning）

这是共享空间最强大的能力之一。假设模型在训练时只见过“猫”和“狗”的EEG-文本配对。现在给它一段“兔子”的EEG信号，它从未见过“兔子”这个词。

-   **在共享空间里**：模型可以将这段EEG信号映射到共享空间，然后发现它离“兔子”这个词的文本向量最近（因为“兔子”和“猫”、“狗”在语义上相似，都在“动物”这个区域）。
-   **结果**：模型可以**零样本**地识别出“兔子”，即使它从未在训练数据中见过“兔子”的EEG信号。

**如果不共享空间，模型只能识别训练集中出现过的类别，无法泛化到新类别。**

#### 3. 实现多模态融合与推理（Multimodal Fusion & Reasoning）

这是更高级的应用。在共享空间里，你可以将多种模态的信息融合在一起，进行更复杂的推理。

-   **例子**：一段EEG信号显示受试者正在思考“苹果”，同时一张图片显示了一个红色的水果。在共享空间里，模型可以将EEG向量和图片向量进行融合（例如，取平均或拼接），然后推理出：“受试者正在思考一个红色的苹果。”
-   **如果不共享空间**，EEG和图片的信息无法融合，模型只能分别处理，无法进行联合推理。

#### 4. 简化模型架构与训练

-   **统一处理**：一旦EEG和文本都在同一个空间里，模型就可以用**同一套Transformer**来处理它们。这大大简化了模型架构，不需要为每个模态设计独立的处理模块。
-   **共享参数**：LLM的词嵌入层可以被EEG信号复用。这意味着EEG信号可以“借用”LLM强大的语言理解能力，而无需从头学习语言知识。



---

### 潜在问题与权衡

当然，共享空间并非完美无缺：

| 问题 | 说明 | 解决方案 |
| :--- | :--- | :--- |
| **信息损失** | 将EEG信号映射到文本空间，可能会丢失EEG信号中特有的信息（如频段能量、相位同步等）。 | 使用更强大的投影器（如Q-Former），保留更多原始信息。 |
| **模态主导** | 如果文本数据量远大于EEG数据量，共享空间可能会被文本模态“主导”，EEG信号的特征被淹没。 | 在训练时平衡两种模态的数据量，或使用模态特定的正则化。 |
| **对齐精度** | 对比学习只能保证“大致对齐”，无法保证“精确对齐”。EEG向量可能落在文本空间中两个相近词之间。 | 使用更精细的损失函数（如三重损失），或结合生成损失进行微调。 |

